# Looks To The Sun — interactive demo

Launch LTTS in a temporary Colab runtime. Run the cells from top to bottom.

A T4 runtime is recommended; CPU also works for `SmolLM-135M`.

## 1. Clone the repositories


In [ ]:
!git clone --depth 1 https://github.com/seruva19/LTTS.git
# LTTS loads modules from the sibling checkout.
!git clone --depth 1 https://github.com/seruva19/LTTS-modules.git
%cd LTTS


## 2. Install Python dependencies

Keep Colab's PyTorch build and install the remaining dependencies.

In [ ]:
# Keep Colab's PyTorch and NumPy builds.
!grep -vE '^(--extra-index-url|torch|torchvision|torchaudio|repeng)' requirements.txt > /tmp/ltts_req.txt
!pip install -q -r /tmp/ltts_req.txt
!pip install -q 'gguf>=0.6.0' 'scikit-learn>=1.4.0'
!pip install -q --no-deps repeng==0.4.0
print('Python dependencies installed.')

## 3. Build the frontend

Build the Vue app served by LTTS.

In [ ]:
!npm install --no-audit --no-fund --loglevel=error
!npm run build-client
print('Frontend built into dist/.')

## 4. Start the server

Start LTTS and open it inside this notebook.

In [ ]:
import os, subprocess, time, urllib.request
from google.colab import output

PORT = 8000
env = dict(os.environ)
env.update(
    LTTS_HOST='127.0.0.1',
    LTTS_PORT=str(PORT),
    LTTS_SKIP_MODULE_VENVS='1',  # use the base env; skip per-module venvs in the demo
    MPLBACKEND='agg',
)

server = subprocess.Popen(['python', 'ltts.py'], cwd=os.getcwd(), env=env)

# Wait for the server to come up (importing torch/transformers takes a moment).
health = f'http://127.0.0.1:{PORT}/'
for i in range(90):
    try:
        if urllib.request.urlopen(health, timeout=2).status == 200:
            print(f'Server is up after ~{i*2}s.')
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Server did not start - check the cell output above.')

import json as _json
with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/ltts/modules', timeout=10) as _r:
    _modules = _json.load(_r).get('modules', {})
with open('../LTTS-modules/registry.json', encoding='utf-8') as _r:
    _expected = {item['name'] for item in _json.load(_r)['modules']}
_missing = sorted(_expected - set(_modules))
if _missing:
    raise RuntimeError(f'Modules failed to load: {_missing}')
print(f'All {len(_modules)} analysis modules loaded.')

# Colab's authenticated proxy avoids short-lived public tunnel URLs.
output.serve_kernel_port_as_iframe(PORT, height=900)

## 5. How to test it

Use the embedded app above, then:

1. Load `HuggingFaceTB/SmolLM-135M` from **Models -> Load model...**
2. Expand a transformer layer and attach `attn_visualizer` or another module.
3. Enter a prompt in **Model Input** and click **Run**.
4. Check that the module artifact appears in the layer panel.

---
To stop the demo, enable `STOP_DEMO` below and run that cell.

In [ ]:
STOP_DEMO = False # @param {type:"boolean"}
if STOP_DEMO:
    server.terminate()
    print('Server stopped.')
else:
    print('Demo remains available. Enable STOP_DEMO and run this cell to stop it.')